# TP 2 — Premiers pas PySpark

**Big Data Engineering — Master 1 — DMI/FST/UCAD — Prof. Samba Ndiaye**

Ce notebook guide les parties B (WordCount), C (fil rouge) et D (Spark UI)
du TP 2. Complétez les cellules marquées `À COMPLÉTER`, exécutez tout de bout
en bout, puis poussez le notebook **avec ses sorties** dans `notebooks/`.

**Livrable** : ce notebook, avec le WordCount **commenté ligne à ligne**, les
observations de la Spark UI, et la comparaison Spark vs Pandas.

> Réflexe d'ingénieur : on ne dit pas « c'est lent », on dit « 4,2 s pour
> 50 000 lignes ». Mesurez tout.

## 0. Vérification de l'environnement

PySpark s'exécute sur la JVM : un **JDK 17** est requis. Si la cellule
échoue, revenez à la partie A du TP (`java -version`).

In [1]:
import sys, platform
import pyspark
print("Python  :", sys.version.split()[0], "-", platform.system())
print("PySpark :", pyspark.__version__)

Python  : 3.13.5 - Windows
PySpark : 4.1.2


## 1. Créer la SparkSession

`master("local[*]")` exécute Spark dans ce notebook, sur **tous les cœurs**
de la machine. La Spark UI démarre sur http://localhost:4040 — ouvrez-la.

In [2]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("TP2-WordCount")
         .master("local[*]")
         .getOrCreate())

print("Spark", spark.version)
print("Coeurs vus :", spark.sparkContext.defaultParallelism)
print("Spark UI   :", spark.sparkContext.uiWebUrl)

Spark 4.1.2
Coeurs vus : 4
Spark UI   : http://192.168.56.1:4040


## Exercice B — WordCount

Préparez d'abord un fichier texte dans `data/discours.txt` (un discours, un
article — au moins quelques centaines de mots). La cellule ci-dessous compte
les mots. **Commentez chaque ligne** : c'est le cœur du livrable.

In [ ]:
# === À COMPLÉTER ===
from pathlib import Path
from pyspark.sql.functions import explode, split, lower, col

# Chemin du fichier texte à analyser
chemin = "../data/discours.txt"
if not Path(chemin).exists():
    chemin = "data/discours.txt"

# Lire le fichier texte : une ligne = une ligne du discours
texte = spark.read.text(chemin)

# Transformer chaque ligne en mots, les mettre en minuscule,
# supprimer les chaînes vides, puis compter les occurrences.
mots = (texte
        # split(...) découpe la chaîne en mots selon les espaces
        .select(explode(split(lower(col("value")), r"\s+")).alias("mot"))
        # supprimer les mots vides créés par la séparation
        .filter(col("mot") != "")
        # groupBy + count réalise une agrégation, donc un shuffle
        .groupBy("mot")
        .count())

# Afficher les 15 mots les plus fréquents
mots.orderBy(col("count").desc()).show(15)


### B — Observer (répondez en markdown)

1. Quelles lignes sont des **transformations** ? Laquelle est l'**action** ?
2. Où se situe le **shuffle** ?
3. Si vous relancez `mots.orderBy(...).show()`, pourquoi tout est-il
   recalculé ?

*Vos réponses :* …

## Exercice C — Recharger le fil rouge avec Spark

On reprend les fichiers du fil rouge, cette fois avec Spark. `inferSchema`
demande à Spark de **deviner** les types (pratique mais coûteux : une passe
de lecture en plus).

In [ ]:
# === À COMPLÉTER ===
orders = (spark.read
          .option("header", True)
          .option("inferSchema", True)
          .csv("data/orders.csv"))

events = spark.read.json("data/events.json")   # JSON Lines

orders.printSchema()
print("orders :", ...)        # nombre de lignes
events.printSchema()

### C — Explorer sans tout rapatrier

`show(n)` et `count()` sont des actions ; `select`, `filter`, `groupBy`
décrivent seulement le plan. **Ne jamais** faire `events.collect()`.

In [ ]:
# === À COMPLÉTER ===
orders.select("order_id", "statut", "canal").show(5)

livrees = orders.filter(col("statut") == "livree")
print("livrees :", ...)

orders.groupBy("statut").count().show()
# ... compter par canal, trie par count decroissant
...

## Exercice C.3 — Spark vs Pandas : mesurez

Chronométrez Spark sur `orders.csv`, comparez à votre mesure Pandas du TP 1
(même fichier, échelle 0.1). Lequel gagne sur ce petit volume ?

In [ ]:
# === À COMPLÉTER ===
import time

t0 = time.perf_counter()
n = spark.read.option("header", True).csv("data/orders.csv").count()
t_spark = time.perf_counter() - t0
print("Spark : %.2f s pour %d lignes" % (t_spark, n))

# Reportez ici votre temps Pandas du TP 1 :
t_pandas = ...
print("Pandas (TP1) : %s s" % t_pandas)

## Exercice D — Lire la Spark UI

Ouvrez http://localhost:4040. Cette cellule donne les partitions ; le reste
s'observe **dans le navigateur** (onglets Jobs, Stages, SQL/DataFrame).

In [ ]:
# === À COMPLÉTER ===
print("Partitions orders :", orders.rdd.getNumPartitions())
print("Coeurs disponibles :", spark.sparkContext.defaultParallelism)

# Relancez le WordCount pour le retrouver dans l'onglet Jobs :
mots.orderBy(col("count").desc()).show(5)

### D — Relevés (complétez en markdown)

- Nombre de stages du WordCount : …
- Shuffle Read / Write du stage d'agrégation : …
- Nombre de tasks par stage / nombre de partitions : …
- Tasks en parallèle vs nombre de cœurs : …

## 5. Avant de pousser

Vérifiez : WordCount commenté ligne à ligne, observations Spark UI
renseignées, comparaison Spark/Pandas chiffrée. Puis :

```bash
git add notebooks/TP2_wordcount.ipynb
git commit -m "TP2 : WordCount PySpark, fil rouge, Spark UI"
git push
```

Pensez à **arrêter la session** en fin de travail : `spark.stop()`.

In [ ]:
spark.stop()